# 📕 Bài 2 — Phân loại & Metrics

**Tuần 10** · ⭐ **Notebook quan trọng nhất Phase 04**

> Những khái niệm trong bài này quay lại **nguyên vẹn** ở Phase 07 khi bạn viết eval cho LLM. Đừng học vẹt — hiểu cho kỹ.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.figsize"] = (11, 4)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

MAU = ["#4269d0", "#efb118", "#3ca951", "#ff725c", "#a463f2"]
MAU_CHINH, MAU_NHAN = MAU[0], MAU[3]

DATA = Path("../../../data")
if not (DATA / "churn.csv").exists():
    DATA = Path("data")

df = pd.read_csv(DATA / "churn.csv")

# BO cot ro ri ngay tu dau - se giai thich ky o bai 3
df = df.drop(columns=["da_goi_tong_dai_huy"])

print(f"{len(df)} khach hang   |   ty le churn: {df['churn'].mean():.1%}")
df.head(3)

## 1. ⚠️ Accuracy là cái bẫy

In [ ]:
ty_le_churn = df["churn"].mean()

print("Model NGU NGỐC: 'không ai rời bỏ cả'")
print(f"  Accuracy = {1 - ty_le_churn:.1%}")
print("  Bắt được số khách sắp rời bỏ: 0")
print()
print("Nghe như một thành tựu. Thực tế: hoàn toàn vô dụng.")

> 🔑 **Accuracy chỉ có nghĩa khi các lớp cân bằng.**
>
> Với gian lận thẻ tín dụng (0.1% gian lận), model luôn đoán "sạch" đạt accuracy **99.9%** và vô dụng tuyệt đối.
>
> Bất cứ báo cáo nào khoe accuracy mà không nói tỷ lệ lớp đều đang che giấu điều gì đó.

## 2. Train một model phân loại

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

X = df.drop(columns=["churn", "khach_id"])
y = df["churn"]

cot_so = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
cot_chu = [c for c in X.columns if c not in cot_so]

tien_xu_ly = ColumnTransformer([
    ("so", Pipeline([("dien", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler())]), cot_so),
    ("chu", OneHotEncoder(handle_unknown="ignore"), cot_chu),
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y      # stratify: giữ tỷ lệ lớp
)

model = Pipeline([("pre", tien_xu_ly),
                  ("model", LogisticRegression(max_iter=1000))]).fit(X_train, y_train)

print(f"Tỷ lệ churn — train: {y_train.mean():.1%}   test: {y_test.mean():.1%}")
print("(nhờ stratify nên hai con số này gần bằng nhau)")

## 3. `predict` vs `predict_proba`

In [ ]:
nhan = model.predict(X_test)
xac_suat = model.predict_proba(X_test)[:, 1]

print("predict       :", nhan[:8])
print("predict_proba :", xac_suat[:8].round(3))

> 💡 **Hầu như luôn dùng `predict_proba`.** Xác suất cho bạn bốn thứ mà `predict` vứt bỏ:
> 1. Chọn **ngưỡng** theo bài toán kinh doanh
> 2. **Xếp hạng** khách hàng theo rủi ro (gọi 100 người rủi ro nhất)
> 3. Tính **chi phí kỳ vọng**
> 4. **Ghép nhiều model** bằng cách trung bình xác suất
>
> `.predict()` nén tất cả thông tin đó thành một bit 0/1.

## 4. Confusion matrix

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

cm = confusion_matrix(y_test, nhan)
tn, fp, fn, tp = cm.ravel()

print(f"  TN = {tn:>4}   đoán ở lại, đúng là ở lại")
print(f"  FP = {fp:>4}   đoán rời bỏ, thật ra ở lại   -> TỐN TIỀN khuyến mãi thừa")
print(f"  FN = {fn:>4}   đoán ở lại, thật ra rời bỏ   -> MẤT KHÁCH")
print(f"  TP = {tp:>4}   đoán rời bỏ, đúng là rời bỏ")

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=["Ở lại", "Rời bỏ"]).plot(
    ax=ax, cmap="Blues", colorbar=False)
ax.grid(visible=False)
plt.title("Confusion matrix (ngưỡng 0.5)")
plt.show()

**Mẹo nhớ tên gọi** — đọc theo góc nhìn của model:
- Chữ **thứ hai** (P/N) = model đoán gì
- Chữ **đầu** (True/False) = model đoán đúng hay sai

`False Positive` = model nói "có" nhưng **sai** → báo động giả
`False Negative` = model nói "không" nhưng **sai** → bỏ sót

> **Hai loại sai này có giá khác nhau.** Đó là toàn bộ lý do ta cần nhiều hơn một con số.

## 5. Precision & Recall

In [ ]:
from sklearn.metrics import (accuracy_score, precision_score,
                             recall_score, f1_score, roc_auc_score)

print(f"  Accuracy  {accuracy_score(y_test, nhan):.4f}   <- nhớ: đoán bừa đã được "
      f"{1 - y_test.mean():.4f}")
print(f"  Precision {precision_score(y_test, nhan):.4f}   trong số ta BÁO rời bỏ, "
      f"{precision_score(y_test, nhan):.0%} đúng")
print(f"  Recall    {recall_score(y_test, nhan):.4f}   trong số THẬT SỰ rời bỏ, "
      f"ta bắt được {recall_score(y_test, nhan):.0%}")
print(f"  F1        {f1_score(y_test, nhan):.4f}")
print(f"  ROC-AUC   {roc_auc_score(y_test, xac_suat):.4f}")

```
              TP                          TP
Precision = ────────           Recall = ────────
            TP + FP                      TP + FN
            ↑                            ↑
      mọi thứ ta BÁO là 1          mọi thứ THẬT SỰ là 1
```

| | Câu hỏi | Nhớ bằng |
|---|---|---|
| **Precision** | Trong số ta báo, bao nhiêu % đúng? | *"Báo có chuẩn không?"* |
| **Recall** | Trong số thật sự có, ta bắt được bao nhiêu %? | *"Có bỏ sót ai không?"* |

**Hai cách gian lận để đạt điểm cao:**
- Precision = 1.0 dễ đạt: chỉ báo "1" cho **một** mẫu chắc chắn nhất → recall ≈ 0
- Recall = 1.0 dễ đạt: báo "1" cho **tất cả** → precision = tỷ lệ lớp dương

> Vì thế **không bao giờ báo cáo một mình một chỉ số**.

## 6. ⭐ Ngưỡng — cái nút bạn phải tự vặn

In [ ]:
nguongs = np.arange(0.05, 0.96, 0.05)
bang = []
for ng in nguongs:
    yp = (xac_suat >= ng).astype(int)
    bang.append({
        "nguong": ng,
        "precision": precision_score(y_test, yp, zero_division=0),
        "recall": recall_score(y_test, yp),
        "f1": f1_score(y_test, yp),
    })
bang = pd.DataFrame(bang)

plt.plot(bang["nguong"], bang["precision"], lw=2, color=MAU[0], label="Precision")
plt.plot(bang["nguong"], bang["recall"], lw=2, color=MAU[2], label="Recall")
plt.plot(bang["nguong"], bang["f1"], lw=2, color=MAU[3], label="F1")
plt.axvline(0.5, color="gray", ls="--", alpha=.6)
plt.text(0.51, 0.05, "mặc định 0.5", fontsize=9, color="gray")
plt.xlabel("ngưỡng"); plt.ylabel("điểm")
plt.title("Đánh đổi Precision ↔ Recall")
plt.legend(frameon=False)
plt.show()

Hai đường **cắt nhau**: đẩy recall lên thì precision xuống. Không có bữa trưa miễn phí.

> **0.5 không thiêng liêng.** Nó chỉ là mặc định của `.predict()`. Ngưỡng đúng phụ thuộc vào chi phí của FP so với FN — một quyết định **kinh doanh**, không phải kỹ thuật.

## 7. ⭐⭐ Chọn ngưỡng bằng TIỀN

In [ ]:
CHI_PHI_FP = 200_000      # khuyến mãi nhầm cho khách không định bỏ
CHI_PHI_FN = 3_000_000    # mất hẳn một khách hàng

chi_phi = []
for ng in nguongs:
    yp = (xac_suat >= ng).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(y_test, yp).ravel()
    chi_phi.append(fp_ * CHI_PHI_FP + fn_ * CHI_PHI_FN)

chi_phi = np.array(chi_phi)
tot_nhat = nguongs[chi_phi.argmin()]

plt.plot(nguongs, chi_phi / 1e6, lw=2, color=MAU_CHINH, marker="o", ms=4)
plt.axvline(tot_nhat, color=MAU_NHAN, ls="--", lw=2,
            label=f"rẻ nhất: ngưỡng {tot_nhat:.2f}")
plt.axvline(0.5, color="gray", ls=":", label="mặc định 0.5")
plt.xlabel("ngưỡng"); plt.ylabel("tổng chi phí (triệu VND)")
plt.title("Chi phí kinh doanh theo ngưỡng")
plt.legend(frameon=False)
plt.show()

cp_mac_dinh = chi_phi[np.argmin(np.abs(nguongs - 0.5))]
print(f"  Ngưỡng mặc định 0.5 : {cp_mac_dinh/1e6:>8.1f} triệu")
print(f"  Ngưỡng tối ưu {tot_nhat:.2f}  : {chi_phi.min()/1e6:>8.1f} triệu")
print(f"  TIẾT KIỆM           : {(cp_mac_dinh - chi_phi.min())/1e6:>8.1f} triệu")

> 🎯 **Đây là kỹ năng phân biệt junior với senior.**
>
> Junior báo cáo: *"F1 = 0.62"*. Sếp không hiểu và không quan tâm.
> Senior báo cáo: *"Chỉnh ngưỡng từ 0.5 xuống 0.25 tiết kiệm 180 triệu/tháng"*. Sếp phê duyệt ngay.
>
> **Quy trình:**
> 1. Hỏi người phụ trách nghiệp vụ: *một FP tốn bao nhiêu? một FN tốn bao nhiêu?*
>    (Họ thường chưa bao giờ được hỏi câu này và sẽ phải suy nghĩ thật.)
> 2. Quét ngưỡng, tính chi phí
> 3. Trình bày biểu đồ trên
> 4. Để **họ** chọn điểm cân bằng

> ⚠️ Trong thực tế, ngưỡng phải chọn trên tập **validation**, không phải test — chọn trên test là một dạng overfitting lên tập test.

## 8. ROC-AUC

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve, average_precision_score

fpr, tpr, _ = roc_curve(y_test, xac_suat)
prec, rec, _ = precision_recall_curve(y_test, xac_suat)

fig, (a1, a2) = plt.subplots(1, 2)

a1.plot(fpr, tpr, lw=2, color=MAU_CHINH,
        label=f"AUC = {roc_auc_score(y_test, xac_suat):.3f}")
a1.plot([0, 1], [0, 1], ls="--", color="gray", label="đoán mò (0.5)")
a1.set_xlabel("False Positive Rate"); a1.set_ylabel("True Positive Rate (Recall)")
a1.set_title("ROC curve"); a1.legend(frameon=False)

a2.plot(rec, prec, lw=2, color=MAU[2],
        label=f"AP = {average_precision_score(y_test, xac_suat):.3f}")
a2.axhline(y_test.mean(), ls="--", color="gray",
           label=f"đoán mò ({y_test.mean():.2f})")
a2.set_xlabel("Recall"); a2.set_ylabel("Precision")
a2.set_title("Precision-Recall curve"); a2.legend(frameon=False)

plt.tight_layout(); plt.show()

**AUC** = xác suất model chấm một khách churn ngẫu nhiên điểm **cao hơn** một khách không churn ngẫu nhiên.

```
1.0   hoàn hảo        0.70  tạm được
0.85  tốt             0.5   đoán mò
                      <0.5  tệ hơn đoán mò 🚨 (thường do đảo nhãn)
```

**Ưu điểm lớn nhất:** AUC **không phụ thuộc ngưỡng** — nó đo chất lượng của *xác suất*, nên so sánh model rất công bằng.

> ⚠️ Khi dữ liệu **cực kỳ** mất cân bằng (< 1% lớp dương), ROC-AUC vẫn đẹp một cách gây hiểu nhầm. Lúc đó nhìn **PR-AUC** (average precision) — đường mò của nó nằm ở tỷ lệ lớp dương, nên nó phản ánh trung thực hơn.

## 9. Xử lý mất cân bằng

In [ ]:
from sklearn.metrics import classification_report

can_bang = Pipeline([
    ("pre", tien_xu_ly),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
]).fit(X_train, y_train)

p_cb = can_bang.predict(X_test)

print("MẶC ĐỊNH:")
print(classification_report(y_test, nhan, target_names=["Ở lại", "Rời bỏ"], digits=3))
print("class_weight='balanced':")
print(classification_report(y_test, p_cb, target_names=["Ở lại", "Rời bỏ"], digits=3))

`class_weight="balanced"` phạt nặng hơn khi model sai ở lớp hiếm → **recall tăng mạnh, precision giảm**. Nó tương đương với việc hạ ngưỡng, chỉ là làm ở tầng loss thay vì tầng quyết định.

**Bốn cách xử lý mất cân bằng, theo thứ tự nên thử:**

| Cách | Lưu ý |
|---|---|
| **Đổi metric** (bỏ accuracy) | Luôn làm đầu tiên, không tốn gì |
| **`class_weight="balanced"`** | Một tham số, thử ngay |
| **Chỉnh ngưỡng** | Không cần train lại |
| **Resampling (SMOTE…)** | Chỉ trên train, dễ gây leakage |

> Người mới hay nhảy ngay vào SMOTE. Ba cách đầu giải quyết được phần lớn trường hợp và ít rủi ro hơn nhiều.

## 10. Tổng kết — quy trình đánh giá phân loại

```
① Nhìn TỶ LỆ LỚP trước         → biết accuracy có dùng được không
② Tính baseline                 → "luôn đoán lớp phổ biến" được bao nhiêu?
③ predict_proba, không predict  → giữ lại thông tin
④ Nhìn confusion matrix         → hiểu model sai KIỂU gì
⑤ Precision + Recall (cả hai)   → không rút gọn thành một số quá sớm
⑥ AUC để so sánh model          → không phụ thuộc ngưỡng
⑦ Chọn ngưỡng bằng CHI PHÍ      → nói chuyện được với người ra quyết định
```

> 🔗 **Liên hệ Phase 07:** eval cho LLM dùng **chính những khái niệm này**. Khi bạn đo *"bao nhiêu % câu trả lời trích dẫn đúng nguồn"* — đó là precision. *"Bao nhiêu % thông tin cần thiết được tìm ra"* — đó là recall. Chỉ khác là đầu ra không xác định tuyệt đối.

## ✍️ Bài tập

Bài tập tuần này bắt bạn **tự code lại mọi metric** — sau đó bạn sẽ không bao giờ đọc một bảng metric mà không hiểu nữa.

```bash
pytest tests/phase04/test_ex02.py -v
```